# Molo: compute_chem_properties + train quick check

This notebook is a minimal scaffold to verify that `compute_chem_properties_*` works, and to show how to kick off training from Python.

In [1]:
import os, sys, json
from pathlib import Path

# Ensure repo root is on sys.path
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)


Repo root: /work/nvme/bfuy/rgao7/Molo


In [2]:
import torch
import os
import argparse
import numpy as np
import types
import builtins
import sklearn.svm._classes as new_svm_classes
import sys
import socket, platform
print("FLASK HOST:", socket.gethostname(), platform.uname())


sys.modules['sklearn.svm.classes'] = new_svm_classes

os.environ['CUDA_VISIBLE_DEVICES'] = ''
safe_types = [
    argparse.Namespace,
    np.ndarray,
    np.dtype,
    np.core.multiarray._reconstruct,
    np.dtype('float64').__class__,  
    set, slice, tuple, list, dict, float, int, str, bytes,
    type, types.SimpleNamespace,
    builtins.object,
]

print("Loading ADMET model...")
with torch.serialization.safe_globals(safe_types):
    from admet_ai import ADMETModel
    admet_model = ADMETModel(num_workers=8)


FLASK HOST: dt-login04.delta.ncsa.illinois.edu uname_result(system='Linux', node='dt-login04.delta.ncsa.illinois.edu', release='5.14.0-427.91.1.el9_4.x86_64', version='#1 SMP PREEMPT_DYNAMIC Wed Sep 17 10:08:59 EDT 2025', machine='x86_64')
Loading ADMET model...


/u/rgao7/.conda/envs/molo/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.b

## Compute properties (local)
Pick a few SMILES and run the local property computation.

In [3]:
from props.properties import compute_chem_properties_batch

sys.modules['sklearn.svm.classes'] = new_svm_classes

os.environ['CUDA_VISIBLE_DEVICES'] = ''
safe_types = [
    argparse.Namespace,
    np.ndarray,
    np.dtype,
    np.core.multiarray._reconstruct,
    np.dtype('float64').__class__,  
    set, slice, tuple, list, dict, float, int, str, bytes,
    type, types.SimpleNamespace,
    builtins.object,
]
# Example SMILES
source_smiles = ['CCO']
target_smiles = [['CCN', 'CCCl']]

src_props, tgt_props = compute_chem_properties_batch(source_smiles, target_smiles, task='bbbp+drd2+herg+qed', admet_model=admet_model)
print('Source properties:')
print(src_props)
print('Target properties:')
print(tgt_props)


model ensembles:   0%|          | 0/2 [00:00<?, ?it/s]














model ensembles:  50%|█████     | 1/2 [00:09<00:09,  9.30s/it]














model ensembles: 100%|██████████| 2/2 [00:18<00:00,  9.45s/it]


Source properties:
[{'valid': True, 'smiles': 'CCO', 'qed': 0.40680796565539457, 'drd2': 0.005215657636036644, 'bbbp': 0.9718312621116638, 'herg': 0.01059387483401224}]
Target properties:
[[{'valid': True, 'smiles': 'CCN', 'qed': 0.40623709538988323, 'drd2': 0.008926437181009215, 'bbbp': 0.9394632339477539, 'herg': 0.07479139678180217, 'sim': 0.3333333333333333}, {'valid': True, 'smiles': 'CCCl', 'qed': 0.37198554725482313, 'drd2': 0.004326296356393497, 'bbbp': 0.9993820548057556, 'herg': 0.2816188868135214, 'sim': 0.3333333333333333}]]


/u/rgao7/.conda/envs/molo/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 0.21.3 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
[19:47:00] DEPRECATION WARNING: please use MorganGenerator
[19:47:00] DEPRECATION WARNING: please use MorganGenerator
[19:47:00] DEPRECATION WARNING: please use MorganGenerator


## Training entrypoint
You can launch training via `train_grpo.py` or `train_ppo.py`.
This cell shows how to call `train_grpo.main()` with a config path.
Adjust `config_path` as needed.

In [ ]:
import sys
from train_grpo import main as train_grpo_main

# Point to your config
config_path = 'configs/train_config_mistral_bdeq.yaml'

# Simulate CLI args for train_grpo
sys.argv = ['train_grpo.py', '--config', config_path]

# This will start training; comment out if you only want to test imports
# train_grpo_main()
print('Ready to train with config:', config_path)
